# Automated AI form filler

This project explains how to create an AI agent that automatically completes forms. The project involves integrating AI models with web technologies to automate the process of filling out forms based on provided data.

![AI Form Flow](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/d28a41926f852f670c3e16b502343d2058a14978/AI%20from%20flow.png)

The following steps outline a simple way that a computer program can complete a form for you:

It finds the form and determines what needs to be completed: First, the program looks at a webpage form to see where you need to enter information, such as your name or address.

It retrieves the information from the files: The program reads the files that you have, such as PDF files or Word documents, to find the information that needs to go into the form.

It enters the information in the correct spots on the form: The program takes the information it found and enters it in the correct places on the webpage form, like typing your name where it asks for 'Name'.

So, basically, the program helps you complete forms online by using the information from your documents without you having to type it all in.

## Setting Up the Environment

First, let's set up the environment by running following lines in the new terminal.

Run the following commands to receive the project, then it rename it to proper name, and it moves the files into the appropriate directory, by running following commands.

Install the requirements for the project.

This installation approximately takes 5 minutes to get completed.

When complete, the structure of the files should look similar to the following image.

![File Structure](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/68a7f0f8d2ef6f965dd3e80ec15052afb8c7ecc0/file%20structure.jpg)

##### Next, click and open Auto_fill_AI.py

## Auto Filler API

### Functions in the Python files:

- get_llm(): This function initializes an AI language model that is provided by IBM Watson (LLM). It's used later to understand questions and to provide relevant answers from the documents.
- process_data(): This function controls the reading of documents and turning them into a structured format (such as a list of key points or summaries) that the AI can use. It also creates a searchable database so that the AI can quickly find the information needed for each form field.
- get_form_field_descriptions(): This function reads an HTML form and identifies all of the different places where information must be entered, such as the name or address. It finds these fields by their labels and IDs.
- filling_form(): With the information from documents processed and ready to use, this function asks the AI to complete the form. It uses the database to find the right information for each field.
- handle_tax_form_data(): This part of the code is for the web server. It takes the completed form from the previous steps and gets it ready to be sent back to a webpage in a structured way that the webpage can understand (this is usually done in JSON format).

### Process data:

- The Process data section shows that the documents are broken down into chunks for processing.
- These chunks are then converted into embeddings, which are numerical representations of text that the AI can understand and work with.
- These embeddings are stored in a vector database, which is a special kind of database that is optimized for operations involving vectors (such as searching for the closest matching piece of information).

### The main flow:

- The Python code retrieves information from documents and processes it.
- Then, it reads and understands what information the HTML form asks for.
- Finally, it sends the relevant information to complete each field of the form.

So, the entire flow is about taking documents, processing them into a form that AI can use, and then using that processed information to complete forms automatically. The information flow then goes from the Python back end to the JavaScript front end, which likely manages the user interface or handles user interactions on the webpage.

## Libraries and Initiating LLMs

### Stage 1: Importing Libraries

#### Code Snippet

In [ ]:
# Import required libraries from various modules.
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM
from ibm_watson_machine_learning.metanames import GenTextParamsMetaNames as GenParams
from ibm_watson_machine_learning.foundation_models import Model

# langchain library for embeddings, text splitting, and conversational retrieval.
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts.prompt import PromptTemplate

# Document loader and vector store modules for processing PDFs.
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.vectorstores import FAISS

# BeautifulSoup for parsing HTML content.
from bs4 import BeautifulSoup

# Flask for web server and CORS for cross-origin resource sharing.
from flask import Flask, jsonify, render_template
from flask_cors import CORS

### Stage 2: Initializing the AI Model

#### Code Snippet

In [ ]:
#------------> Function to initialize and retrieve the language model from IBM Watson. <------------
def get_llm():
    # Credentials for accessing IBM Watson services.
    my_credentials = {
        "url": "enter-the-url-of-the-server",
    }

    # Parameters for controlling the generation of text by the model.
    params = {
        GenParams.MAX_NEW_TOKENS: 256, # Maximum tokens generated per run.
        GenParams.TEMPERATURE: 0.0,    # Controls randomness in generation.
    }

    # Initialize the model with specified ID and credentials.
    LLAMA2_model = Model(
        model_id='enter-your-model-id', 
        credentials=my_credentials,
        params=params,
        project_id="enter-your-project-id")

    # Create and return the Watson Language Model.
    llm = WatsonxLLM(model=LLAMA2_model)
    return llm


Testing your code regularly is essential. You can do this by executing the script and obtaining a response from the LLM instance. Use the .predict method to input your prompt and then display the result. For instance, try llm.predict("Tell me how it's like to live in Toronto?") to see what the model says.

To run your code,

In [ ]:
python3.11 Auto_filler_AI.py

The following image is how your result will look like.

![result 1](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/aedf0902c61a428c57f2636109146169eb8117b3/result%201.jpg)

## Data Processing

### Stage 3: Data processing and creating vector database

Creating a vector database and using a ranking (or retrieval) algorithm mechanism are essential steps for efficiently handling and searching through large amounts of text data, especially in language processing applications like the AI auto form filler you're working with.

##### Vector Database

- Purpose: A vector database stores information in a format that a computer can process quickly. It turns text into mathematical vectors, which are lists of numbers that represent the text's meaning.
- Why it's needed: When dealing with lots of documents or large texts, finding the specific piece of information that you need to fill out a form can be like looking for a needle in a haystack. A vector database allows the AI to search through these 'haystacks' incredibly fast to find the 'needle' – the exact bit of information required.
- Benefits: This approach is much faster than reading through every document each time you need to find something. It's also more accurate because the AI understands the context of the information it's searching for, thanks to the mathematical properties of vectors.

In this auto form filler project, the vector database helps the AI understand what it's reading in the documents, and the ranking mechanism ensures that the information that is used to complete the forms is as relevant as possible to the questions asked by each form field.

Processing the data involves loading, splitting, and indexing documents, making them searchable and retrievable for the AI model.

##### Loading documents: 
    PDF files from the "info" directory are loaded into the system. I set PyPDFDirectoryLoader to load PDF files from a directory.

##### Splitting text:
    The text in these documents is divided into smaller pieces, or "chunks," to make it easier to handle. RecursiveCharacterTextSplitter splits the documents into smaller chunks for easier processing.

##### Creating embeddings:
- The HuggingFaceInstructEmbeddings line refers to using a pretrained model from Hugging Face (a popular NLP library) to create embeddings.
- The specific model, "sentence-transformers/all-MiniLM-L6-v2", is designed to convert sentences into embeddings. This means that it reads the chunks of text and turns each one into a vector.
- Each vector captures the semantic meaning of the text chunk, including context and the relationship of words within that chunk.

##### Indexing for retrieval:
- The FAISS.from_documents line is about taking those vectors and putting them into a database called FAISS (Facebook AI Similarity Search).
- FAISS is designed to efficiently search through these vectors to find the best matches for any given query. This makes it much faster to find relevant pieces of text later when filling out forms.

Creating an embedding is the process of turning chunks of text from PDF files into vectors that capture their meaning, enabling the AI to search and retrieve information quickly.

### Code Snippet

In [ ]:
def process_data():
    # Load PDF documents from a specified directory.
    loader = PyPDFDirectoryLoader("info")
    docs = loader.load()

    # Split the text from documents for better processing.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
    texts = text_splitter.split_documents(docs)

    # Create embeddings for the text data.
    embeddings = HuggingFaceInstructEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # Index the documents using FAISS for efficient retrieval.
    db = FAISS.from_documents(texts, embeddings)
    return db


Then, run the Python file.

In [ ]:
python3.11 Auto_filler_AI.py

The following image is how the result will look like.

![result 2](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/67b2fada781fb7b17acfdf5724f8eaff30e2a35d/result%202.jpg)

## Get the fields of the form

### Stage 4: Form field identification

##### Why:
    To automate form completion, you first need to know what fields are present in the form and their characteristics.

##### How:
- Parse HTML: Read the HTML file and use BeautifulSoup to parse it.
- Extract form fields: Identify form elements like input boxes, drop-down menus, and text areas, and extract their labels and IDs for identification.

In this case, the following image shows the simple tax form, which is an HTML file inside of the folder templates.

![Result 3](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/e9114b332cef96b3780e019d358267110ce95e6b/result%203.jpg)

The get_form_field_descriptions function is designed to read an HTML file and extract information about form fields that are present in it. The following information is a step-by-step breakdown.

##### Opening and reading the HTML file:
- The function takes a file path (html_file_path) as input.
- It opens the specified HTML file and reads its content into a variable called html_content.

##### Parsing HTML content:
- The content of the HTML file is then parsed using BeautifulSoup, a Python library for pulling data out of HTML and XML files.
- This parsing turns the HTML content into a format that can be easily navigated and searched programmatically.

##### Finding form fields:
- The function searches through the parsed HTML for form fields. Specifically, it looks for HTML elements like input, select, and textarea, which are common in web forms.
- Each of these elements represents a different type of input field on a form.

##### Extracting field information:
- For each form field found, the function collects relevant information.
- It first tries to find a label element that is associated with the field, which typically provides a human-readable description of the field's purpose (like "Name" or "Email Address").
- If a label is not found, the function then looks for a placeholder or name attribute in the field as a fallback to use as a label.
- It also captures the field's ID or name attribute, which is often used to uniquely identify the field in the form.

##### Storing field data:
- The collected information (label and ID or name) for each field is stored in a dictionary (field_data).
- This dictionary is then added to a list (field_info), which contains the data for all of the fields found in the form.

##### Returning the data:
- Finally, the function returns the field_info list, which now holds dictionaries of label and ID/name pairs for each form field in the HTML file.

In summary, this function parses an HTML form and extracts structured information about its fields, which can then be used for various purposes like automating form filling.

### Code Snippet

In [ ]:
def get_form_field_descriptions(html_file_path):
    with open(html_file_path, 'r') as file:
        html_content = file.read()

    soup = BeautifulSoup(html_content, 'html.parser')

    # Find and process all form fields in the HTML.
    form_fields = soup.find_all(['input', 'select', 'textarea'])
    field_info = []
    for field in form_fields:
        field_data = {}

        # Extract label text or use placeholder/name as a fallback.
        label = soup.find('label', {'for': field.get('id')})
        if label:
            field_data['label'] = label.get_text().strip().rstrip(':')
        else:
            placeholder = field.get('placeholder')
            name = field.get('name')
            description = placeholder if placeholder else name
            if description:
                field_data['label'] = description.strip()

        # Include the ID or name of the field in the data.
        field_id = field.get('id') or field.get('name')
        if field_id:
            field_data['id'] = field_id

        # Add complete field data to the list.
        if 'label' in field_data and 'id' in field_data:
            field_info.append(field_data)

    return field_info


Test the function get_form_field_descriptions and print the output.

In [ ]:
python3.11 Auto_filler_AI.py

You should get a dictionary.

![Result 4](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/c6feb60ebbf2fb572d82e96e502113683e763eff/result%204.jpg)

## Generating response based on the documents and the HTML fields

### Stage 5: Generating Responses

##### Why:
    This stage is where the AI model is used to complete the form based on the content of the loaded documents.

Before delving into code, I want to briefly dicuss retrieval-augmented generation (RAG).

#### Ranking/retrieval-augmented generation:
- Purpose: The retrieval mechanism helps to decide which pieces of information are the most relevant to the question or prompt you're dealing with.
- Why it's needed: When the AI searches the vector database, it might find many pieces of information that could be the answer. The ranking algorithm sorts these results to show the best match at the top, much like how search engines show you the most relevant results first.
- Benefits: This ensures that the AI fills out the form with the most accurate information available. It saves time by reducing the need to manually check whether the AI chose the right details.

The filling_form function in this Python script is designed to automatically complete form fields using information extracted from documents. The following is a detailed explanation of how it works.

##### Initializing language model and data processing:
- The function begins by initializing a language model (llm) using the get_llm() function. This model is capable of understanding and generating natural language.
- It also calls process_data(), which likely processes documents (like PDF files) to make them searchable and extracts relevant information for use.

##### Preparing for form filling:
- The function receives form_fields_info, a list of dictionaries where each dictionary contains information about a form field (like its label and ID).
- An empty list structured_responses is created to store the responses for each form field.

##### Creating prompts for each field:
- For each field in form_fields_info, the function generates a prompt. This prompt is a question for the language model, asking it to provide the information needed for that specific form field.
- The prompt is engineered using the field's label and ID to specify what information is required.

##### Setting up a conversational retrieval chain:
- A conversational retrieval chain is set up using the language model (llm) and the document database (db). This chain handles the task of finding the correct answer from the processed documents based on the prompt.
- db.as_retriever likely converts the document database into a format that can be used to retrieve specific pieces of information. The search_kwargs={'k': 4} parameter suggests that the top 4 most relevant pieces of information are retrieved for consideration.

##### Retrieving and storing responses:
- For each field, the conversational chain is executed with the given prompt. This process involves the AI model searching the document database to find the most relevant answer.
- The response is extracted from the result, trimmed of any extra whitespace, and then added to structured_responses along with the field's information.

##### Returning the filled fields:
- After processing all form fields, the function returns structured_responses, which now contains the original field information and the corresponding responses (answers) from the AI model.

In [ ]:
def filling_form(form_fields_info):
    # Initialize the language model and data processing tools.
    llm = get_llm()
    db = process_data()

    structured_responses = []
    for field in form_fields_info:
        # Create a specific prompt for each form field.
        prompt = f"Based on the document, what is the '{field['label']}'? Provide only the required information for the field ID '{field['id']}'."

        # Set up a conversational chain to retrieve and generate responses.
        conversation_chain = ConversationalRetrievalChain.from_llm(
            llm=llm,
            retriever=db.as_retriever(search_kwargs={'k': 4}),
            condense_question_prompt=PromptTemplate(input_variables=[], template=prompt),
        )

        # Get the response for each field.
        result = conversation_chain({"question": prompt, "chat_history": []})
        structured_responses.append({**field, "response": result['answer'].strip()})

    return structured_responses


Test the function and evaluate the output. To run it, you must first get the data for the form fields (previous stage) and pass it to filling_form.

In [ ]:
python3.11 Auto_filler_AI.py

Feel free to play with the prompt template and change it to make it more accurate output.

You should get a response like the folowing code. This is a standard way to represent structured data in JSON format, making it suitable for various applications like web APIs, configuration files, and data storage. Theis information is passed to JavaScript to complete the HTML form.

![Result 5](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/757dd754a0da34c6343d0ae5e3263b4f24b8d743/result%205.jpg)

## Final step: put the mechanism into server and deploy it

### Why do you need flask?
    To enable your JavaScript code to dynamically complete a form using information from a an LLM, you must create an intermediary server-side component, such as an application that is built with Flask or Django. This server-side application acts as a bridge between your client-side JavaScript and the Python-based LLM. The following information explains how you can structure this setup:

##### Server-side application (Python):
- Develop a server-side application using a web framework like Flask. This application runs on a server and is responsible for handling HTTP requests.
- Your Python application communicates with the LLM to process data and generate responses.

##### API endpoint creation:
- Within your server-side application, set up an API endpoint. This is a specific URL where your server listens for incoming requests from the client-side JavaScript.
- For instance, you might create an endpoint like /api/get-form-data that's dedicated to processing requests related to form filling.

##### Client-side JavaScript interaction:
- Your JavaScript code, running in the user's browser, makes requests to the server-side API endpoint when it needs data to complete the form. The JavaScript code is available in the static folder.
- This can be done using JavaScript's fetch API or with other HTTP client libraries to send requests to your server.

##### Processing requests with LLM:
- When the server-side Python application receives a request at the API endpoint, it interacts with the LLM to process the request. This might involve using the LLM to interpret the request, search for the necessary data, or generate the appropriate text for form fields.

##### Sending data back to JavaScript:
- After processing the request, the server-side application sends a response back to the JavaScript code. This response contains the data or information needed to complete the form.
- The response is usually formatted as JSON, which is easy for JavaScript to parse and use to update the web page's content.

##### Form filling in JavaScript:
- After the JavaScript code receives the response from the server, it uses this data to dynamically complete the form fields on the webpage.

In summary, you set up a Python-based server application that interfaces with the LLM. This server application exposes an API endpoint that your client-side JavaScript can call to request data for form filling. Upon receiving the data from the server, the JavaScript then populates the form fields accordingly.

### Stage 6: Deploying the web server

##### Why:
The web server acts as an interface between the user and the AI form filler, allowing users to access the function through API calls.

##### How:
- Set up Flask: Create a Flask app and configure CORS for cross-origin requests.
- Create API endpoint: Define a route in Flask that is called to get the filled form data.
- Run server: Start the Flask server to make the API accessible.

### Code Snippet

In [ ]:
app = Flask(__name__)
CORS(app)  # Enable cross-origin requests.

# Define route for the home page.
@app.route('/')
def home():
    return render_template('styled_tax_form.html')

# Define API route to retrieve form data.
@app.route('/api/get_tax_form_data', methods=['GET'])
def get_tax_form_data():
    data_from_form = get_form_field_descriptions("templates/styled_tax_form.html")
    structured_responses = filling_form(data_from_form)

    # Convert responses to a JSON format for the frontend.
    response_data = {field['id']: field['response'] for field in structured_responses}
    return jsonify(response_data)

# Run the Flask application if this script is executed directly.
if __name__ == '__main__':
    app.run(debug=True, port=5055)


Add the code snippet to your Auto_filler_AI.pyfile, and run it.

In [ ]:
python3.11 Auto_filler_AI.py

Because you have configured the server to use port=5055, the output message indicates that the server is operational on the default path at port 5055. While the server is running, I aim to launch the web application. To facilitate this, the Skills Network team has provided a plug-in. Follow the provided instructions to access and use the app effectively.

![Result 6](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/0ff75ad8beee931cfa5ed5c908ef70720c0496b2/result%206.jpg)

Then, run the application:

![Result 7](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/b38195586e5780d5939805591c70a386c7facf61/result%207.jpg)

You will have the online form. Click AI Auto-Fill Tax Form and enjoy how AI complets the form for you!

![Result 8](https://raw.githubusercontent.com/Lenifer15/Automated-AI-Form-Filler/2cd20d8cd0af7ee882c0c5281da916f674f8d641/result%208.jpg)

### Conclusion:

An automated AI form filler streamlines repetitive data entry by intelligently extracting and populating information into forms.
It reduces manual effort, minimizes human errors, and significantly improves efficiency.
The system leverages AI to understand input data and adapt to different form structures.
This makes it highly useful for tasks like registrations, applications, and data processing workflows.
Overall, it demonstrates how AI agents can automate real-world tasks and enhance productivity.

#### Aut